# Wind forecast plots for C st.

This notebook reads the newest `data/wind_points_out_*.csv` file, filters to `site_name = "C st."`, and plots:
- 10m wind speed (m/s) for all models
- 10m wind direction (deg FROM, unwrapped) for all models

## Using your `.venv` as the Jupyter kernel
From your project root:
```bash
source .venv/bin/activate
pip install ipykernel jupyter matplotlib pandas numpy
python -m ipykernel install --user --name wind-venv --display-name "Python (wind-venv)"
jupyter notebook
```
Then in Jupyter: **Kernel → Change kernel → Python (wind-venv)**.


In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

DATA_DIR = Path("data")
SITE_NAME = "C st."   # must match the site_name in data/locations.csv
USE_LOCAL_TIME = True  # True: use valid_time_local_6h; False: use valid_time_utc

def find_newest_output_csv(data_dir: Path) -> Path:
    files = sorted(data_dir.glob("wind_points_out_*.csv"))
    if not files:
        raise FileNotFoundError(f"No wind_points_out_*.csv found in {data_dir.resolve()}")
    return files[-1]

def unwrap_degrees(series_deg: pd.Series) -> pd.Series:
    """Unwrap 0–360 degrees to a continuous series for plotting."""
    rad = np.deg2rad(series_deg.to_numpy(dtype=float))
    unwrapped = np.unwrap(rad)
    return pd.Series(np.rad2deg(unwrapped), index=series_deg.index)


In [ ]:
# Load newest output CSV
csv_path = find_newest_output_csv(DATA_DIR)
df_all = pd.read_csv(csv_path)
print("Using:", csv_path)
df_all.head()

In [ ]:
# Filter to successful rows and the target site
df = df_all.copy()
df = df[df.get("status", "").eq("ok")]
df = df[df["site_name"] == SITE_NAME].copy()

if df.empty:
    print("Available site_name values:")
    print(sorted(df_all["site_name"].dropna().unique().tolist()))
    raise ValueError(f"No rows for site_name='{SITE_NAME}' with status=='ok' in {csv_path.name}.")

time_col = "valid_time_local_6h" if USE_LOCAL_TIME else "valid_time_utc"
df[time_col] = pd.to_datetime(df[time_col])

# Aggregate just in case there are duplicates per (time, model)
df = (
    df.groupby([time_col, "model"], as_index=False)
      .agg(
          wind_speed_mps=("wind_speed_mps", "mean"),
          wind_dir_deg_from=("wind_dir_deg_from", "mean"),
      )
)

df.head()

In [ ]:
# Pivot to wide for plotting
speed_wide = df.pivot(index=time_col, columns="model", values="wind_speed_mps").sort_index()
dir_wide = df.pivot(index=time_col, columns="model", values="wind_dir_deg_from").sort_index()

speed_wide.head()

In [ ]:
# Plot wind speed
plt.figure(figsize=(12, 6))
for col in speed_wide.columns:
    plt.plot(speed_wide.index, speed_wide[col], label=col)
plt.xlabel(f"Valid time ({'local' if USE_LOCAL_TIME else 'UTC'})")
plt.ylabel("10 m wind speed (m/s)")
plt.title(f"Wind speed forecasts at {SITE_NAME} (all models)")
plt.legend(loc="best", fontsize=9)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Plot wind direction (unwrapped to avoid 0/360 jumps)
plt.figure(figsize=(12, 6))
for col in dir_wide.columns:
    s = dir_wide[col].dropna()
    if s.empty:
        continue
    s_unwrap = unwrap_degrees(s)
    plt.plot(s_unwrap.index, s_unwrap.values, label=col)
plt.xlabel(f"Valid time ({'local' if USE_LOCAL_TIME else 'UTC'})")
plt.ylabel("10 m wind direction (deg FROM, unwrapped)")
plt.title(f"Wind direction forecasts at {SITE_NAME} (all models)")
plt.legend(loc="best", fontsize=9)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()